In [16]:
import newkernelo as newker
import kernelo as oldker
import numpy as np
import time
import logging
logging.getLogger().setLevel(logging.INFO)

## Global parameters

In [17]:
gamma_type = 'Full'
sigma_type = 'Diag'
seed = 12345678

# initialisation parameters
gllim_em_iteration = 50
gllim_em_floor = 1e-12
gmm_kmeans_iteration = 1
gmm_em_iteration = 1
gmm_floor = 1e-12
nb_experiences = 10

# training parameters
train_max_iteration = 300
train_ratio_ll = 1e-5
train_floor = 1e-12

##### Generate random dataset

In [18]:
L, D, K, N = 4, 9, 5, 1000

x_gen = np.random.rand(N,L)
y_gen = np.random.rand(N,D)

#### Generate sobol Test Model dataset

In [19]:
L, D, K, N = 4, 9, 5, 100

# Create "physical" model
# dir_path = os.path.dirname(os.path.realpath(__file__))
# print(dir_path+"/pytest/models")
# physical_model = oldker.ExternalModelConfig("RPVModel", "rpv_model", dir_path + "/pytest/models").create()
physical_model = oldker.TestModelConfig().create()

# Create StatModel
covariances = np.random.uniform(0, 0.0001, physical_model.get_D_dimension())
stat_model = oldker.GaussianStatModelConfig("sobol", physical_model, covariances, 12345).create()

# Initialize and train GLLIM model
print("Generating dataset")
x_gen, y_gen = stat_model.gen_data(N)
print(x_gen.shape)
print(y_gen.shape)

Generating dataset
(100, 4)
(100, 9)


#### Generate sobol RPV model dataset

In [20]:
L, D, K, N = 3, 71, 10, 100

# Create "physical" model
# dir_path = os.path.dirname(os.path.realpath(__file__))
# print(dir_path+"/pytest/models")
physical_model = oldker.ExternalModelConfig("RPVModel", "rpv_model", "../../pytest/models").create()

# Create StatModel
covariances = np.random.uniform(0, 0.0001, physical_model.get_D_dimension())
stat_model = oldker.GaussianStatModelConfig("sobol", physical_model, covariances, 12345).create()

# Initialize and train GLLIM model
print("Generating dataset")
x_gen, y_gen = stat_model.gen_data(N)
print(x_gen.shape)
print(y_gen.shape)

Generating dataset
(100, 3)
(100, 71)


## OLD GLLiM

In [21]:
# Create GLLIM model, including its initialization and training configuration
learningConfig = oldker.EMLearningConfig(train_max_iteration, train_ratio_ll, train_floor)
# learningConfig=oldker.GMMLearningConfig(train_max_iteration, train_ratio_ll, train_floor)
# initConfig = oldker.MultInitConfig(seed=123456789, nb_iter_EM=10, nb_experiences=10, gmmLearningConfig=oldker.GMMLearningConfig(15, 10, 1e-12))
initConfig = oldker.MultInitConfig(seed=seed, nb_iter_EM=gllim_em_iteration, nb_experiences=nb_experiences, gmmLearningConfig=oldker.GMMLearningConfig(gmm_kmeans_iteration, gmm_em_iteration, gmm_floor))
gllim_old= oldker.GLLiM(D, L, K, gamma_type, sigma_type, initConfig, learningConfig)


In [22]:

print("Initializing GLLIM model")
gllim_old.initialize(x_gen, y_gen)
gllim_old_params_0 = gllim_old.exportModel() # you can export your gllim model parameters

print("Training model")
gllim_old.train(x_gen, y_gen)
gllim_old_params = gllim_old.exportModel() # you can export your gllim model parameters


INFO:root:Start Multi initialization
INFO:root:Initialisation : 1
INFO:root:	Generate GMM means
INFO:root:	Generate GMM covariance matrices
INFO:root:	Train the GMM model
INFO:root:	Compute Initial theta vector of the GLLiM model
INFO:root:	Train the initial GLLiM model
INFO:root:	Current log likelihood : 108.211038, Best log likelihood : 108.211038


Initializing GLLIM model


INFO:root:Initialisation : 2
INFO:root:	Generate GMM means
INFO:root:	Generate GMM covariance matrices
INFO:root:	Train the GMM model
INFO:root:	Compute Initial theta vector of the GLLiM model
INFO:root:	Train the initial GLLiM model
INFO:root:	Current log likelihood : 102.299832, Best log likelihood : 108.211038
INFO:root:Initialisation : 3
INFO:root:	Generate GMM means
INFO:root:	Generate GMM covariance matrices
INFO:root:	Train the GMM model
INFO:root:	Compute Initial theta vector of the GLLiM model
INFO:root:	Train the initial GLLiM model
INFO:root:	Current log likelihood : 121.425216, Best log likelihood : 121.425216
INFO:root:Initialisation : 4
INFO:root:	Generate GMM means
INFO:root:	Generate GMM covariance matrices
INFO:root:	Train the GMM model
INFO:root:	Compute Initial theta vector of the GLLiM model
INFO:root:	Train the initial GLLiM model
INFO:root:	Current log likelihood : 107.922058, Best log likelihood : 121.425216
INFO:root:Initialisation : 5
INFO:root:	Generate GMM me

Training model


### Get theta_0 from oldker

In [23]:
# theta_0 = newker.GLLiMParameters(L,D,K, "full", "diag")

# print(np.array(gllim_parameters_0.Pi).shape)
# print(np.array(gllim_parameters_0.A).shape)
# print(np.array(gllim_parameters_0.C).shape)
# print(np.array(gllim_parameters_0.Gamma).shape)
# print(np.array(gllim_parameters_0.B).shape)
# print(np.array(gllim_parameters_0.Sigma).shape)

# print(np.array(gllim_parameters_0.Pi))
# print(np.transpose(np.array(gllim_parameters_0.A), (1,2,0)).shape)
# print(np.array(gllim_parameters_0.C).shape)
# print(np.transpose(np.array(gllim_parameters_0.Gamma), (1,2,0)).shape)
# print(np.array(gllim_parameters_0.B).shape)
# print(np.transpose(np.array(gllim_parameters_0.Sigma), (1,2,0)).shape)

# theta_0.Pi = np.copy(np.array(gllim_parameters_0.Pi))
# theta_0.A = np.copy(np.transpose(np.array(gllim_parameters_0.A), (1,2,0)))
# theta_0.C = np.copy(np.array(gllim_parameters_0.C))
# # theta_0.Gamma = np.copy(np.transpose(np.array(gllim_parameters_0.Gamma), (1,2,0)))
# theta_0.Gamma = np.copy(gllim_parameters_0.Gamma)
# theta_0.B = np.copy(np.array(gllim_parameters_0.B))
# # theta_0.Sigma = np.copy(np.transpose(np.array(gllim_parameters_0.Sigma), (1,2,0)))
# Sigma_diag = np.zeros((K,D))
# for k in range(K):
#     Sigma_diag[k,:] = np.diag(gllim_parameters_0.Sigma[k])
# theta_0.Sigma = Sigma_diag

# print(np.sum(theta_0.Pi))
# gllim.setParams(theta_0)

## NEW GLLiM

In [26]:
n_hidden_variables = 0
L = L + n_hidden_variables
new_gllim = newker.GLLiM(K,D,L,gamma_type.lower(), sigma_type.lower(), n_hidden_variables)

verbose = 1
X_gen = x_gen.T
Y_gen = y_gen.T

new_gllim.initialize(X_gen, Y_gen, gllim_em_iteration, gllim_em_floor, gmm_kmeans_iteration, gmm_em_iteration, gmm_floor, nb_experiences, seed, verbose)
new_gllim_params_0 = new_gllim.getParams()

new_gllim.train(X_gen, Y_gen, train_max_iteration, train_ratio_ll, train_floor, verbose)
new_gllim_params = new_gllim.getParams()


INFO: GLLiM Parameters initialized
INFO: GLLiM dimensions are (L=4, D=71, K=10)
INFO: GLLiM constraints are :
	gamma_type = 'full',
	sigma_type = 'diag'.


INFO: Start Initialization
INFO: Initialisation : 1
INFO: 	Generate GMM means
INFO: 	Generate GMM covariance matrices
INFO: 	Train the GMM model
INFO: 	Compute Initial theta vector of the GLLiM model from GMM
INFO: 	Train the initial GLLiM model
INFO: Start GLLiM-EM Training
INFO: Iteration : 1, log likelihood : -inf
INFO: Iteration : 2, log likelihood : -inf
INFO: Iteration : 3, log likelihood : -inf
INFO: Iteration : 4, log likelihood : -inf
INFO: Iteration : 5, log likelihood : -inf
INFO: Iteration : 6, log likelihood : -inf
INFO: Iteration : 7, log likelihood : -inf
INFO: Iteration : 8, log likelihood : -inf
INFO: Iteration : 9, log likelihood : -inf
INFO: Iteration : 10, log likelihood : -inf
INFO: Iteration : 11, log likelihood : -inf
INFO: Iteration : 12, log likelihood : -inf
INFO: Iteration : 13, log likelihood : -inf
INFO: Iteration : 14, log likelihood : -inf
INFO: Iteration : 15, log likelihood : -inf
INFO: Iteration : 16, log likelihood : -inf
INFO: Iteration : 17, log lik

In [ ]:
print(gllim_old_params_0.Pi)
print(new_gllim_params_0.Pi)

print("\n")
print(gllim_old_params.Pi)
print(new_gllim_params.Pi)

print("\n")
print(gllim_old_params.B)
print(new_gllim_params.B)

[0.07 0.12 0.17 0.04 0.08 0.1  0.11 0.21 0.07 0.03]
[[0.1  0.11 0.07 0.08 0.1  0.1  0.1  0.22 0.08 0.04]]


[0.07 0.12 0.17 0.04 0.08 0.1  0.11 0.21 0.07 0.03]
[[0.1  0.11 0.07 0.08 0.1  0.1  0.1  0.22 0.08 0.04]]


[[ 5.53439947e-01  5.61419823e-01  6.32304801e-01  7.60394798e-01
   9.51451087e-01  1.20072794e+00  1.46498713e+00  1.46497865e+00
   1.20081155e+00  9.51329713e-01]
 [ 7.60413137e-01  6.32287656e-01  5.61527461e-01  5.53139655e-01
   1.46121513e-01  2.18222803e-01  2.85247125e-01  3.66820513e-01
   4.79054194e-01  6.41717778e-01]
 [ 8.77546958e-01  1.20091505e+00  1.58786164e+00  1.87371252e+00
   1.67107524e+00  1.46494932e+00  1.33919364e+00  1.36970534e+00
   3.82747370e+00  3.57479199e+00]
 [ 3.54327257e+00  2.43275112e+00  1.67114244e+00  1.12418859e+00
   7.60413654e-01  5.24282003e-01  3.67084763e-01  2.53431540e-01
   1.60058481e-01  6.45770446e-02]
 [-6.53583940e-02 -3.13542452e-01 -1.72265460e+00 -7.04164670e-01
  -2.87148804e-01 -6.51276325e-02  8.56533798e-02 

In [ ]:
# Y_gen = np.array(y_gen[:5].T)
# pred = new_gllim.inverseDensities(Y_gen)
# print(pred.meanPredResult.mean)
# print(x_gen[:5])
# print("\n")


predicator = oldker.PredictionConfig(2, 2, 1e-10, gllim_old).create()
for i in range(5):
    print(x_gen[i])

    prediction = predicator.predict(y_gen[i], np.zeros(D))
    print(prediction.meansPred.mean)

    pred = new_gllim.inverseDensities(y_gen[i])
    print(pred.meanPredResult.mean)
    print("\n")


[0.5 0.5 0.5]
[0.62571059 0.57057257 0.49222271]
INFO: Inverse theta
[[0.5684554  0.53923587 0.5174954 ]]


[0.75 0.25 0.25]
INFO: Construct the GMM of the inverse conditional model
INFO: Compute the weighted mean of the means in the mixture
INFO: Compute the weighted covariance of the covariances in the mixture
[0.75265876 0.2044838  0.23439707]
[[0.7565452  0.21240831 0.23605227]]


[0.25 0.75 0.75]
INFO: Inverse theta
INFO: Construct the GMM of the inverse conditional model
INFO: Compute the weighted mean of the means in the mixture
INFO: Compute the weighted covariance of the covariances in the mixture
[0.84880503 0.75378505 0.89956332]
[[0.06732838 0.30506709 0.61954235]]


[0.375 0.375 0.625]
INFO: Inverse theta
INFO: Construct the GMM of the inverse conditional model
INFO: Compute the weighted mean of the means in the mixture
INFO: Compute the weighted covariance of the covariances in the mixture
[0.36474009 0.35872506 0.62201735]
INFO: Inverse theta
INFO: Construct the GMM of t

In [ ]:
pred.centerPredResult.means

array([], shape=(0, 0), dtype=float64)

In [ ]:
insights = new_gllim.getInsights()

In [ ]:
print(insights.time)
print(type(insights.time))
print(type(insights.initialisation.start_time))

0:00:01
<class 'datetime.timedelta'>
<class 'datetime.datetime'>


In [ ]:
insights.log_likelihood

array([[      -inf],
       [2.21903657],
       [2.21903657],
       [      -inf],
       [      -inf],
       [      -inf],
       [      -inf],
       [      -inf],
       [      -inf],
       [      -inf],
       [      -inf],
       [      -inf],
       [      -inf],
       [      -inf],
       [      -inf],
       [      -inf],
       [      -inf],
       [      -inf],
       [      -inf],
       [      -inf],
       [      -inf],
       [      -inf],
       [      -inf],
       [      -inf],
       [      -inf],
       [      -inf],
       [      -inf],
       [      -inf],
       [      -inf],
       [      -inf],
       [      -inf],
       [      -inf],
       [      -inf],
       [      -inf],
       [      -inf],
       [      -inf],
       [      -inf],
       [      -inf],
       [      -inf],
       [      -inf],
       [      -inf],
       [      -inf],
       [      -inf],
       [      -inf],
       [      -inf],
       [      -inf],
       [      -inf],
       [     